In [ ]:
from datasets import load_dataset, load_from_disk
import pandas as pd
import json
import evaluate
import tiktoken

dataset = load_dataset('EdinburghNLP/xsum')
gpt_dataset = load_from_disk('xsum/xsum_test_gpt4_turbo')
test = dataset['test']
gpt_test = gpt_dataset['test']

In [ ]:


def cal_words_num(text):
    return len(text.split())

def cal_tokens_num(text):
    enc = tiktoken.encoding_for_model('gpt-4')
    token = enc.encode(text)
    return len(token)


with open('xsum/sorted_test_gpt4_turbo.json') as f:
    data = json.load(f)

gpt_data = pd.DataFrame(data)

test = pd.DataFrame(test)
test.replace('\n', ' ', regex=True, inplace=True)
gpt_data.replace('\n', ' ', regex=True, inplace=True)

mturk = pd.DataFrame({
    "original_article": test['document'],
    "Summary_1": test['summary'],
    "Summary_2": gpt_data['summary'],
})

mturk.to_csv('xsum/Mturk_csv/gpt4_vs_Xsum_Full.csv', index=False)


In [ ]:
# Build an index list for each document value in test
index_map = {}
for i, d in enumerate(test):
    doc = d['document']
    if doc not in index_map:
        index_map[doc] = []
    index_map[doc].append(i)

# Helper: return the next index for a given document
def get_next_index(doc, iters):
    if doc in iters and len(iters[doc]) > 0:
        return iters[doc].pop(0)
    return float('inf')  # no more indices left -> return infinity

# Copy the index map for iteration
iters = {doc: indexes.copy() for doc, indexes in index_map.items()}

gpt_test = data

# Sort gpt_test
gpt_test.sort(key=lambda x: get_next_index(x['document'], iters))

with open('Models/LLM_Teached_Pegasus/generated_predictions.json', 'w') as f:
    json.dump(gpt_test, f)

In [ ]:
labels = test['summary']
pred = data['summary']

rouge = evaluate.load('rouge')
bertscore = evaluate.load('bertscore')

def average(list):
    return sum(list)/len(list)

def compute_metrics(pred, labels, rouge, bert_score):
    predictions, labels = pred, labels

    rouge_result = rouge.compute(predictions=predictions, references=labels, use_stemmer=True)
    rouge_result = {k: round(v, 4) for k, v in rouge_result.items()}
    bert_score_result = bert_score.compute(predictions=predictions, references=labels, lang="en")
    del bert_score_result['hashcode']
    bert_score_result = {k: round(average(v), 4) for k, v in bert_score_result.items()}


    return {**rouge_result, **bert_score_result}

res = compute_metrics(pred, labels, rouge, bertscore)

In [ ]:
print(res)

In [ ]:
with open('test_result/Pegasus_vs_xsum.json', 'w') as f:
    json.dump(res, f, indent=4)